# J26-DS-336 Agent 1: width sweeps for Qwen2.5 0.5B / 1.5B / 3B / 7B (FP16)

Plan: `reports/agent1_width/plan_weeks5-6.md`. Rules fixed in advance: `reports/agent1_width/README.md`.

**Before you start**
1. Upload `j26_code.zip` and `j26_data.zip` to Google Drive, folder **`MyDrive/j26_width/`**.
2. Runtime -> Change runtime type -> **A100 GPU** (L4 also works, slower; T4 cannot hold the 7B model).
3. Run cells 1-6 in order. Cell 5 downloads the models (about 22 GB) and checks 5 questions per model.

**After a disconnect:** run cells 1, 2, 3 and 6 again (skip 4 and 5). Every sweep continues where it stopped.
Results and logs are on Drive in `MyDrive/j26_width/results/`. Cell 7 shows the progress; run it whenever you like.

In [ ]:
# 1. GPU and Google Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
for f in ('j26_code.zip', 'j26_data.zip'):
    print(f, 'found' if os.path.exists('/content/drive/MyDrive/j26_width/' + f) else 'MISSING: upload it to MyDrive/j26_width/')

In [ ]:
%%bash
# 2. Unpack code and databases
set -e
rm -rf /content/j26 && mkdir -p /content/j26 && cd /content/j26
unzip -q /content/drive/MyDrive/j26_width/j26_code.zip && unzip -q -o /content/drive/MyDrive/j26_width/j26_data.zip
echo "code commit: $(unzip -z /content/drive/MyDrive/j26_width/j26_code.zip | tail -1)"
echo "databases: $(find data -name '*.sqlite' -o -name '*.db' | wc -l)"


In [ ]:
%%bash
# 3. Install (the GPU's own torch stays; versions pinned to the laptop run: the classifiers were saved with scikit-learn 1.9.1)
cd /content/j26
pip install -q chromadb "spacy==3.8.14" "tiktoken==0.12.0" "onnxruntime==1.29.0" pytest "scikit-learn==1.9.1" joblib fastapi uvicorn httpx sqlalchemy "transformers==5.20.2" "accelerate==1.15.0"
python -m spacy download en_core_web_md -q
python -c "import torch, transformers; import sklearn; print('sklearn', sklearn.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '|', torch.cuda.get_device_name(0))"


In [ ]:
# 4. Tests (width code only)
!cd /content/j26 && python -m pytest -q tests/test_agent1.py -k "width or qwen_local or distraction or transfer" 2>&1 | tail -3

In [ ]:
%%bash
# 5. Smoke check: downloads each model, 5 Chinook questions each. Expect no error and readable accuracy lines.
cd /content/j26
for s in 0.5B 1.5B 3B 7B; do
  echo "== Qwen2.5-$s-Instruct"
  python -m scripts.width_sweep --pairs eval/chinook_pairs.json --local fp16 --hf-model Qwen/Qwen2.5-$s-Instruct --limit 5 --out /content/smoke/$s.jsonl 2>&1 | tail -8
done


In [ ]:
# 6. Start (or continue) all runs in the background. Safe to run again: a second start does nothing.
!cd /content/j26 && nohup bash scripts/colab_runs.sh >> /content/drive/MyDrive/j26_width/runs.log 2>&1 &
print('started; use cell 7 for progress')

In [ ]:
# 7. Progress (run any time)
import glob, os, subprocess
R = '/content/drive/MyDrive/j26_width/results'
total = {'dev': 869, 'train': 1144}
for f in sorted(glob.glob(R + '/*.jsonl')):
    n = sum(1 for _ in open(f))
    print(f'{os.path.basename(f):28} {n:5} / {total[os.path.basename(f).split("_")[0]]}')
print(open('/content/drive/MyDrive/j26_width/runs.log').read()[-500:] if os.path.exists('/content/drive/MyDrive/j26_width/runs.log') else 'no runs.log yet')
print(subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
print('running sweeps:', subprocess.run('pgrep -fc scripts.width_sweep', shell=True, capture_output=True, text=True).stdout.strip())

In [ ]:
# 8. When runs.log says ALL DONE: the summary table at the end of each log.
# Then download MyDrive/j26_width/results/ and copy its files into the project folder results/width/colab/.
!for f in /content/drive/MyDrive/j26_width/results/*.log; do echo "== $f"; grep -v "^[0-9]*/[0-9]*$" "$f" | tail -9; done